In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
result = response.output_parsed

print(result)

questions=['Can I still join the course if I found it late, and do I still have a chance at the certificate?', 'If I start the course now, what do I need to do to be eligible for a certificate?', 'Is it okay to join the course after it has already started, or is that too late?', 'Do late joiners need to submit the project before a certain deadline to get certified?', 'If I’m just discovering the course now, can I still participate and earn the certificate?']


In [15]:
print(result.questions)

['Can I still join the course if I found it late, and do I still have a chance at the certificate?', 'If I start the course now, what do I need to do to be eligible for a certificate?', 'Is it okay to join the course after it has already started, or is that too late?', 'Do late joiners need to submit the project before a certain deadline to get certified?', 'If I’m just discovering the course now, can I still participate and earn the certificate?']


In [16]:
from evaluation_utils import llm_structured

In [17]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it still possible to join the course if I found it late?', 'Can I join the course after it has already started?', 'If I sign up now, can I still get a certificate?', 'What do I need to do to be eligible for the certificate after joining late?', 'Do I have to submit my project before submissions close to receive the certificate?']


In [18]:
usage.input_tokens, usage.output_tokens

(207, 85)

In [20]:
from evaluation_utils import calc_price
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00038250000000000003,
 'total_cost': 0.00053775}

In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it still possible to join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Can I join the course after it has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I sign up now, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate after joining late?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit my project before submissions close to receive the certificate?',
  'document': '74eb249bbf'}]

In [22]:
from evaluation_utils import llm_structured_retry

In [23]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [24]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [25]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [26]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [27]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [28]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07747275

In [29]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07747275

In [30]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [32]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)